In [ ]:
# Re-import necessary packages after code execution environment reset
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import to_rgba

# Define the domain
r = 1.0
theta_vals = np.linspace(0, 2 * np.pi, 100)
x = r * np.cos(theta_vals)
y = r * np.sin(theta_vals)

# Partial derivatives of x(r,θ) and y(r,θ)
def dx_dr(theta): return np.cos(theta)
def dx_dtheta(theta): return -r * np.sin(theta)
def dy_dr(theta): return np.sin(theta)
def dy_dtheta(theta): return r * np.cos(theta)

# Plot setup
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(x, y, label=r"$x = r \cos\varphi$, $y = r \sin\varphi$", color='black')

# Sample a few angles to plot derivatives
sample_thetas = np.linspace(0, 2*np.pi, 12, endpoint=False)
colors = ['royalblue', 'forestgreen']

for theta in sample_thetas:
    x0 = r * np.cos(theta)
    y0 = r * np.sin(theta)
    
    # Arrows for partial derivatives
    dr_vec = np.array([dx_dr(theta), dy_dr(theta)])
    dtheta_vec = np.array([dx_dtheta(theta), dy_dtheta(theta)])
    
    dr_vec /= np.linalg.norm(dr_vec)
    dtheta_vec /= np.linalg.norm(dtheta_vec)

    ax.arrow(x0, y0, 0.3*dr_vec[0], 0.3*dr_vec[1], head_width=0.05, color=colors[0], label=r"$\partial_r$" if theta==sample_thetas[0] else "")
    ax.arrow(x0, y0, 0.3*dtheta_vec[0], 0.3*dtheta_vec[1], head_width=0.05, color=colors[1], label=r"$\partial_\varphi$" if theta==sample_thetas[0] else "")

# Style
ax.set_aspect('equal')
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Partial Derivatives of Coordinate Transformation")
ax.legend()
ax.grid(True)


In [ ]:
# Re-import libraries due to kernel reset
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from pathlib import Path

# Output directory
out_dir = Path("vector_spaces_example")
out_dir.mkdir(parents=True, exist_ok=True)

# Define curves
# Parallel transported: constant vector transported along a spiral curve
def gamma(t):  # spiral in polar coords -> r(t), θ(t)
    r = 1 + 0.5 * t
    theta = 2 * np.pi * t
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return x, y

# Autoparallel (geodesic-like): a curve following the direction of the vector field itself
def delta(t):
    x = t
    y = np.tanh(2 * t)
    return x, y

# Tangent vectors
def gamma_tangent(t):
    # Derivative of gamma(t)
    r = 1 + 0.5 * t
    theta = 2 * np.pi * t
    drdt = 0.5
    dthetadt = 2 * np.pi

    dxdt = drdt * np.cos(theta) - r * np.sin(theta) * dthetadt
    dydt = drdt * np.sin(theta) + r * np.cos(theta) * dthetadt
    return dxdt, dydt

def delta_tangent(t):
    dxdt = 1
    dydt = 2 / (np.cosh(2 * t) ** 2)
    return dxdt, dydt

# Plotting
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# --- Parallel Transport ---
t_vals = np.linspace(0, 1, 100)
xg, yg = gamma(t_vals)
axs[0].plot(xg, yg, label=r"$\gamma(\lambda)$", color='royalblue')
axs[0].set_title("Parallel Transported Curve γ")
axs[0].set_aspect('equal')

# Constant vector to transport
v = np.array([1.0, 0.3])
for t in np.linspace(0, 1, 12):
    x, y = gamma(t)
    axs[0].arrow(x, y, v[0]*0.3, v[1]*0.3, head_width=0.08, head_length=0.12,
                 fc='darkorange', ec='darkorange')

axs[0].set_xlim(-2, 2)
axs[0].set_ylim(-2, 2)
axs[0].set_xlabel("$x$")
axs[0].set_ylabel("$y$")
axs[0].legend()

# --- Autoparallel Transport ---
td_vals = np.linspace(-2, 2, 300)
xd, yd = delta(td_vals)
axs[1].plot(xd, yd, label=r"$\delta(\lambda)$", color='forestgreen')
axs[1].set_title("Autoparallel Curve δ (Geodesic-like)")
axs[1].set_aspect('equal')

for t in np.linspace(-1.8, 1.8, 14):
    x, y = delta(t)
    dx, dy = delta_tangent(t)
    norm = np.sqrt(dx**2 + dy**2)
    axs[1].arrow(x, y, dx / norm * 0.3, dy / norm * 0.3,
                 head_width=0.08, head_length=0.12,
                 fc='crimson', ec='crimson')

axs[1].set_xlim(-2.5, 2.5)
axs[1].set_ylim(-2, 2)
axs[1].set_xlabel("$x$")
axs[1].set_ylabel("$y$")
axs[1].legend()

fig.suptitle("Comparison: Parallel Transport vs Autoparallel (Geodesic-like) Transport", fontsize=14)
fig.tight_layout()
fig.savefig(out_dir / "parallel_vs_autoparallel_transport.pdf")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d
from matplotlib import cm
from pathlib import Path

# Output path
out_dir = Path("plots")
out_dir.mkdir(exist_ok=True)

# === Custom 3D Arrow Class ===
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0, 0), (0, 0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def draw(self, renderer):
        xs, ys, zs = proj3d.proj_transform(*self._verts3d, self.axes.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        super().draw(renderer)

    def do_3d_projection(self, renderer=None):
        _, _, zs = proj3d.proj_transform(*self._verts3d, self.axes.M)
        return np.mean(zs)

# === Utility: Add Text Label to 3D Curve ===
def add_label_3d(ax, x_vals, y_vals, z_vals, label, color, idx=-1, offset=(0.2, 0.2, 0.2)):
    x = x_vals[idx] + offset[0]
    y = y_vals[idx] + offset[1]
    z = z_vals[idx] + offset[2]
    ax.text(x, y, z, label, fontsize=16, color=color, weight='bold')

# === Parameters ===
u = np.linspace(0, np.pi / 2, 100)
v = np.linspace(0, 2 * np.pi, 100)
u, v = np.meshgrid(u, v)

# Sphere coordinates
x = np.sin(u) * np.cos(v)
y = np.sin(u) * np.sin(v)
z = np.cos(u)

# === Geodesic triangle ===
# North pole to equator point A
theta = np.linspace(0, np.pi/2, 50)
x1 = np.sin(theta)
y1 = np.zeros_like(theta)
z1 = np.cos(theta)

# North pole to equator point B
x2 = np.sin(theta) * np.cos(np.pi/4)
y2 = np.sin(theta) * np.sin(np.pi/4)
z2 = np.cos(theta)

# Equator arc from A to B
phi = np.linspace(0, np.pi/4, 50)
x3 = np.cos(phi)
y3 = np.sin(phi)
z3 = np.zeros_like(phi)

# === Create the figure ===
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the sphere
ax.plot_surface(x, y, z, color='lightgrey', alpha=0.9, edgecolor='none')

# Plot equator
phi_eq = np.linspace(0, 2*np.pi, 200)
x_eq = np.cos(phi_eq)
y_eq = np.sin(phi_eq)
z_eq = np.zeros_like(phi_eq)
ax.plot(x_eq, y_eq, z_eq, linestyle='--', color='black', lw=1)

# Plot geodesics
ax.plot(x1, y1, z1, color='blue', lw=2)
ax.plot(x2, y2, z2, color='blue', lw=2)
ax.plot(x3, y3, z3, color='blue', lw=2)

# Add parallel transport vectors (tangent to the curves)
arrow_style = dict(mutation_scale=10, arrowstyle='-|>', color='red', linewidth=1.5)
sample_idx = np.linspace(5, len(theta) - 5, 5).astype(int)

# Curve 1
for i in sample_idx:
    dx = np.cos(theta[i])
    dy = 0
    dz = -np.sin(theta[i])
    ax.add_artist(Arrow3D([x1[i], x1[i] + 0.2 * dx], [y1[i], y1[i] + 0.2 * dy], [z1[i], z1[i] + 0.2 * dz], **arrow_style))

# Curve 2
for i in sample_idx:
    dx = np.cos(theta[i]) * np.cos(np.pi/4)
    dy = np.cos(theta[i]) * np.sin(np.pi/4)
    dz = -np.sin(theta[i])
    ax.add_artist(Arrow3D([x2[i], x2[i] + 0.2 * dx], [y2[i], y2[i] + 0.2 * dy], [z2[i], z2[i] + 0.2 * dz], **arrow_style))

# Curve 3 (equator)
sample_phi = np.linspace(5, len(phi) - 5, 5).astype(int)
for i in sample_phi:
    dx = -np.sin(phi[i])
    dy = np.cos(phi[i])
    dz = 0
    ax.add_artist(Arrow3D([x3[i], x3[i] + 0.2 * dx], [y3[i], y3[i] + 0.2 * dy], [z3[i], z3[i] + 0.2 * dz], **arrow_style))

# Labels
add_label_3d(ax, x1, y1, z1, r"$\gamma$", 'blue', idx=25)
add_label_3d(ax, x2, y2, z2, r"$\delta$", 'blue', idx=25)

# Axis limits and formatting
ax.set_xlim([-1.1, 1.1])
ax.set_ylim([-1.1, 1.1])
ax.set_zlim([0, 1.1])
ax.axis('off')

# Save and show
plt.tight_layout()
fig.savefig(out_dir / "parallel_transport_sphere.pdf")
plt.show()
